# SAM-MedSeg Inference Demo
This notebook demonstrates single-image inference with the improved SAM model for medical image segmentation.

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from models import SAMMedSeg
from utils.visualize import plot_single_result, overlay_mask

%matplotlib inline

In [ ]:
# Load trained model
checkpoint_path = "../checkpoints/best_model.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load(checkpoint_path, map_location=device)
config = checkpoint.get("config", {})

model = SAMMedSeg(
    sam_checkpoint=config.get("sam", {}).get("checkpoint_path", "../checkpoints/sam_vit_b_01ec64.pth"),
    device=torch.device(device),
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
model.merge_lora()

print(f"Model loaded on {device}")
print(f"Best val Dice: {checkpoint.get('best_dice', 'N/A')}")

In [ ]:
def predict(image_path, threshold=0.5):
    img = Image.open(image_path).convert("RGB")
    original = np.array(img)
    img = img.resize((1024, 1024), Image.BILINEAR)
    img_array = np.array(img).astype(np.float32) / 255.0
    img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_tensor)
        logits = output["final_mask"]
        mask = (torch.sigmoid(logits) > threshold).float()
        mask = mask.squeeze().cpu().numpy()
    
    return original, mask

In [ ]:
# Run inference on a test image
image_path = "../data/kvasir-seg/images/cju0qx73cjqw70810hmqgbu15.jpg"
original, mask = predict(image_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(original)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Predicted Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Show overlay
overlay = overlay_mask(
    torch.from_numpy(original).permute(2, 0, 1).float() / 255.0,
    torch.from_numpy(mask),
    alpha=0.4,
    color=(255, 0, 0),
)

plt.figure(figsize=(8, 8))
plt.imshow(overlay)
plt.title("Overlay (Red = Predicted Polyp)")
plt.axis("off")
plt.show()